# DMOSpeech 2: Three Scheduler Revisions in One Comparative Experiment
## Development-Stage Cross-Validation & Baseline Comparison on 100 Development Speakers

**Author:** Rhenel Jhon Sajol (DMOSpeech 2 Project)  
**Specification:** `COLAB_THREE_SCHEDULER_REVISIONS_INSTRUCTIONS.md`  
**Evaluation Scope:** 100 Development Speakers Only (200 Cases: Clean & Noisy15)  
**Candidate Grid:** {2, 3, 4, 5, 6, 8} (1,200 Candidate Records)  
**Status:** **Pre-Registered Development-Only Exploratory Analysis**  

### Objectives & Hypotheses:
1. **Hypothesis:** Can multi-parameter early handover schedulers (Revisions 1, 2, and 3) achieve a superior quality–computation trade-off compared to simple SNR-only policies (SNR-only 2/8, 3/8, 4/8) and the original Candidate A trend rule?
2. **Constraint:** Shared-data experiment on the 100 development speakers. No calibration or test speakers are accessed during threshold selection or rule formulation.
3. **Deliverables:** Unified 11-policy comparison table, paired baseline statistical comparisons, deep case audit, conditional genuine online GPU timing, and master report answering all 6 core thesis questions.

In [ ]:
# Cell 1: Pre-Registration Configuration & Paths Setup
import os
import sys

CONFIG = {
    "experiment_name": "three_scheduler_revisions_dev",
    "manifest_path": "Handover Scheduling/data/splits/development_manifest.json",
    "grid_csv_path": "Handover Scheduling/colab/results/early_handover_dev_gpu/development_expanded_grid.csv",
    "output_dir": "colab/results/three_scheduler_revisions_dev",
    "candidate_steps": [2, 3, 4, 5, 6, 8],
    "step_nfe_map": {2: 9, 3: 11, 4: 13, 5: 15, 6: 17, 8: 21},
    "robust_risk_target": 0.05,  # 5% robust speaker failure development screen
    "n_outer_folds": 5,
    "random_seed": 42,
    "run_online_timing": True,   # Conditional sequential online GPU timing
    "timing_cases": 50,          # Number of cases to benchmark if GPU available
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("[Config Initialized] Experiment Settings:")
for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")


### Cell 2: Colab Environment Logging & Project Setup
Logs allocated GPU hardware, available VRAM, host RAM, Python/PyTorch/CUDA versions, precision, and clones/updates the thesis repository.

In [ ]:
# Cell 2: Hardware Logging & Repository Setup
import platform
import psutil
import subprocess

print("=== Host Hardware & Software Environment ===")
print(f"OS Platform    : {platform.platform()}")
print(f"Python Version : {platform.python_version()}")
print(f"Host CPU Cores : {os.cpu_count()}")
print(f"Host RAM (GB)  : {psutil.virtual_memory().total / (1024**3):.2f} GB")

try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU Model      : {torch.cuda.get_device_name(0)}")
        print(f"GPU VRAM (GB)  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
        print(f"cuDNN Version  : {torch.backends.cudnn.version()}")
except ImportError:
    print("PyTorch not installed in base environment. Proceeding with environment setup...")

# In Google Colab, clone repository and checkout early-handover-dev branch if not present
if not os.path.exists('Shonget-DMOSpeech2') and not os.path.exists('COLAB_THREE_SCHEDULER_REVISIONS_INSTRUCTIONS.md'):
    print("[Setup] Cloning Shonget-DMOSpeech2 repository on branch early-handover-dev...")
    !git clone -b early-handover-dev https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git
    %cd Shonget-DMOSpeech2
    !git checkout early-handover-dev
    !git pull origin early-handover-dev
else:
    !git checkout early-handover-dev
    !git pull origin early-handover-dev

print("[Setup] Environment ready for execution.")


### Cell 3: Required Inputs Loading & 7-Point Mandatory Verification
Validates the 7 formal requirements from Section 3 before any analysis:
1. Exactly 100 dev speaker IDs, 2 conditions per speaker, 6 candidate indices per case.
2. Exactly 1,200 unique records with zero missing quality or decision inputs.
3. Reference values match corresponding index-8 records identically.
4. Failure flags reproduce the original reference-relative definition.
5. NFE equals $2(s+1)+3$ for all candidate indices.
6. Compatibility and provenance verification.
7. Online signal causality check (signals at step $i$ only use information available at step $i$).

In [ ]:
# Cell 3: Load Inputs & Execute 7-Point Verification
import json
import pandas as pd
import numpy as np

# Locate root
if os.path.exists('COLAB_THREE_SCHEDULER_REVISIONS_INSTRUCTIONS.md'):
    ROOT_DIR = os.getcwd()
elif os.path.exists('Shonget-DMOSpeech2'):
    ROOT_DIR = os.path.abspath('Shonget-DMOSpeech2')
    os.chdir(ROOT_DIR)
else:
    ROOT_DIR = os.path.abspath('..')

manifest_file = os.path.join(ROOT_DIR, CONFIG['manifest_path'])
grid_file = os.path.join(ROOT_DIR, CONFIG['grid_csv_path'])

with open(manifest_file, 'r', encoding='utf-8') as f:
    manifest_cases = json.load(f)
df_grid = pd.read_csv(grid_file)

print(f"Loaded Manifest: {len(manifest_cases)} cases")
print(f"Loaded Grid    : {len(df_grid)} candidate evaluations")

# Run 7-Point Verification
sys.path.insert(0, os.path.join(ROOT_DIR, 'Handover Scheduling', 'colab'))
from run_three_scheduler_revisions_dev import validate_inputs

val_report = validate_inputs(df_grid, manifest_cases, CONFIG['output_dir'])
print(f"=== 7-Point Validation Passed: {val_report['validation_passed']} ===")
for check_name, check_data in val_report['checks'].items():
    status = 'PASS' if check_data.get('passed', True) else 'FAIL'
    print(f"  [{status}] {check_name}")


### Cell 4: Baseline Policies Reconstruction & Historical Fold Audit
Reconstructs the original frozen baselines:
- **Fixed Index 8 Reference:** Quality upper bound (21 NFE).
- **SNR-Only 4/8:** Original proposed early-exit baseline (17 NFE).
- **SNR-Only 3/8:** Stronger intermediate fixed early exit (16 NFE).
- **SNR-Only 2/8:** Cheapest fixed early exit (15 NFE).
- **Original SNR + Drift:** Handover at earliest $s \in \{4, 5, 6\}$ with $d_s \le 0.50$, else 8 (17.31 NFE).
- **Candidate A Replay:** Handover at earliest $s \in \{2, 3, 4, 5, 6\}$ with $r_s \le 1.0$, else 8 (16.25 NFE).

In [ ]:
# Cell 4: Build Fast Case Dictionary & Evaluate Baselines
from run_three_scheduler_revisions_dev import (
    build_case_dictionary,
    policy_fixed,
    policy_snr_only,
    policy_original_snr_drift,
    policy_candidate_a,
    evaluate_policy_on_speakers,
)

case_dict = build_case_dictionary(df_grid)
speakers = sorted(list(set(k[0] for k in case_dict.keys())))

baseline_checks = [
    ("Fixed_Index_8", policy_fixed(8)),
    ("SNR_Only_4_8", policy_snr_only(4)),
    ("SNR_Only_3_8", policy_snr_only(3)),
    ("SNR_Only_2_8", policy_snr_only(2)),
    ("Original_SNR_Plus_Drift", policy_original_snr_drift(0.50)),
    ("Candidate_A_Replay_1.0", policy_candidate_a(1.0)),
]

baseline_summary = []
for name, p_fn in baseline_checks:
    res = evaluate_policy_on_speakers(p_fn, speakers, case_dict)
    baseline_summary.append({
        "policy": name,
        "mean_nfe": res["mean_nfe"],
        "nfe_reduction_%": res["nfe_reduction_vs_step8_%"],
        "robust_failures": res["robust_failures"],
        "robust_risk_%": res["robust_failure_rate_%"],
        "clean_wer": res["clean_mean_wer"],
        "clean_sim": res["clean_mean_sim"],
        "exit_distribution": str(res["exit_counts"]),
    })

df_base_audit = pd.DataFrame(baseline_summary)
display(df_base_audit)


### Cell 5: Implement Three Distinct Scheduler Revisions & Verify Online Causality
All revisions retain pre-generation SNR gating (prompt SNR < 20 dB $\to$ Index 8).
For eligible clean speech (SNR $\ge 20$ dB), candidates examine indices $\{2, 3, 4, 5, 6\}$:
1. **Revision 1 (Trend + Absolute Drift):** Stop when $d_i \le a \text{ AND } r_i \le b$.
2. **Revision 2 (Trend + Conditioning Disagreement):** Stop when $r_i \le b \text{ AND } c_i \le c$ (and opposite direction check $c_i \ge c'$).
3. **Revision 3 (Index-Specific Trend Groups):** Stop when $r_i \le b_i$ with groups: $b_2$ for step 2, $b_3$ for step 3, $b_{\text{later}}$ for steps 4, 5, 6.

**Causality Requirement:** At index $i$, no decision function is permitted to read $d_{i+1}$, $r_{i+1}$, or $c_{i+1}$.

In [ ]:
# Cell 5: Policy Implementations & Online Feature Causality Check
from run_three_scheduler_revisions_dev import (
    policy_revision_1,
    policy_revision_2,
    policy_revision_3,
)

# Verify causality: pass a step record containing future nan for step i+1
test_spk = speakers[0]
clean_recs = case_dict[(test_spk, "clean")]

# Test Revision 1, 2, 3 executions
r1_test = policy_revision_1(a=1.0, b=1.0)(clean_recs, is_clean=True)
r2_test = policy_revision_2(b=1.0, c=0.20)(clean_recs, is_clean=True)
r3_test = policy_revision_3(b2=1.0, b3=1.0, b_later=1.0)(clean_recs, is_clean=True)

print(f"Causality test on speaker '{test_spk}':")
print(f"  Revision 1 Clean Exit Step: {r1_test}")
print(f"  Revision 2 Clean Exit Step: {r2_test}")
print(f"  Revision 3 Clean Exit Step: {r3_test}")
print("[PASS] All decision functions evaluate strictly forward in time.")


### Cell 6: Verify Telemetry Completeness Across Shared Candidate Grid
Verifies that all 1,200 grid candidate rows have complete evaluations and signals ($d_2 \dots d_6$, $q_A$, $q_C$, WER, SIM, UTMOS) to avoid redundant synthesis.

In [ ]:
# Cell 6: Candidate Grid Telemetry Audit
signals = ["d_2", "d_3", "d_4", "d_5", "d_6", "q_A", "q_C", "wer", "similarity", "utmos"]
clean_df = df_grid[df_grid["condition"] == "clean"]

missing_telemetry = {}
for sig in signals:
    missing_telemetry[sig] = int(clean_df[sig].isna().sum())

print("Missing telemetry count in 600 clean candidate records:")
for sig, cnt in missing_telemetry.items():
    print(f"  {sig:12s}: {cnt} missing")

assert all(cnt == 0 for cnt in missing_telemetry.values()), "Error: missing telemetry in development grid!"
print("[PASS] All required candidate signals exist; zero audio re-synthesis needed for quality grid.")


### Cell 7: Predefined Parameter Search Grids & 5 Speaker-Disjoint Outer Folds
Sets up deterministic 5-fold cross-validation on the 100 development speakers (`KFold(n_splits=5, shuffle=True, random_state=42)`).
Both clean and noisy15 conditions of a speaker remain strictly in the same fold.

In [ ]:
# Cell 7: Setup Folds and Predefined Search Spaces
from sklearn.model_selection import KFold

kf = KFold(n_splits=CONFIG['n_outer_folds'], shuffle=True, random_state=CONFIG['random_seed'])
spk_arr = np.array(speakers)

print(f"Created {CONFIG['n_outer_folds']} speaker-disjoint folds across {len(speakers)} speakers.")
for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(spk_arr)):
    print(f"  Fold {fold_idx}: Train {len(tr_idx)} speakers | Validation {len(val_idx)} speakers")


### Cell 8: Inner Parameter Selection & Held-Out Outer Validation
Executes the full cross-validation workflow:
1. Evaluates all search grid combinations on the 80 training speakers per fold.
2. Screens candidates for robust risk $\le 5.0\%$ and selects setting minimizing mean NFE.
3. Freezes that fold's selected parameter rule and evaluates on the 20 held-out validation speakers once.
4. Exports `threshold_search_log.csv`, `revision_fold_results.csv`, and `out_of_fold_case_decisions.csv`.

In [ ]:
# Cell 8: Run 5-Fold Cross-Validation Workflow
from run_three_scheduler_revisions_dev import run_revisions_cross_validation

df_spk_folds, df_search, df_folds, df_oof = run_revisions_cross_validation(
    case_dict=case_dict,
    output_dir=CONFIG['output_dir'],
)

print("=== Out-of-Fold Cross-Validation Results Summary ===")
oof_summary = df_folds.groupby('candidate_formula')[['val_robust_risk_%', 'val_mean_nfe']].mean()
display(oof_summary)

print("
=== Selected Parameters Per Fold ===")
display(df_folds[['fold', 'candidate_formula', 'selected_params', 'val_robust_fails', 'val_mean_nfe', 'val_step2_exits', 'val_step3_exits', 'val_step4_exits']])


### Cell 9: Unified 11-Policy Comparison Table & Paired Baseline Statistical Tests
Evaluates all 11 policies on the same 200 cases (100 clean, 100 noisy15):
- Computes speaker-level paired differences against SNR-only 2/8, 3/8, 4/8, Candidate A, and Fixed 8.
- McNemar exact two-tailed tests on discordant failure pairs.
- 95% speaker cluster bootstrap confidence intervals on NFE difference.
- Wilcoxon signed-rank tests.

In [ ]:
# Cell 9: Unified Table & Paired Baseline Comparisons
from run_three_scheduler_revisions_dev import run_all_policy_comparison

df_all_pol, df_paired, df_audit = run_all_policy_comparison(
    case_dict=case_dict,
    df_oof=df_oof,
    output_dir=CONFIG['output_dir'],
)

print("=== Unified 11-Policy Comparison Table ===")
display(df_all_pol[['policy', 'status', 'mean_nfe', 'nfe_reduction_%', 'robust_failures', 'robust_risk_%', 'clean_wer', 'clean_sim', 'step2_exits', 'step3_exits', 'step4_exits', 'step8_exits']])

print("
=== Paired Statistical Comparisons ===")
display(df_paired[['revision_policy', 'baseline_policy', 'rescued_failures', 'harmed_failures', 'net_failure_reduction', 'mcnemar_exact_p_value', 'mean_nfe_difference', 'nfe_diff_95%_ci_lower', 'nfe_diff_95%_ci_upper']])


### Cell 10: Deep-Dive Case Audit (Rescued, Harmed & Unnecessary Waiting)
Examines individual speaker cases to understand where adaptive rules succeed or fail:
- **Rescued Cases:** Cases that failed under a fixed baseline but passed under the adaptive revision.
- **Harmed Cases:** Cases that passed under a fixed baseline but failed under the adaptive revision.
- **Unnecessary Delays:** Cases where early exits (Step 2 or 3) were acoustically safe, but the policy waited until later steps.

In [ ]:
# Cell 10: Case-Level Failure & Waiting Audit
audit_file = os.path.join(CONFIG['output_dir'], 'failure_and_waiting_case_audit.csv')
df_case_audit = pd.read_csv(audit_file)

unnecessary_wait_cases = df_case_audit[df_case_audit['revision_1_unnecessary_waiting'] == True]
print(f"Cases where Step 3 passed but Revision 1 waited unnecessarily: {len(unnecessary_wait_cases)} / 100 speakers")
if len(unnecessary_wait_cases) > 0:
    display(unnecessary_wait_cases[['speaker_id', 'step3_clean_passed', 'Revision_1_OOF_clean_exit', 'SNR_Only_3_8_clean_exit']].head(10))

rob_fail_cases = df_case_audit[(df_case_audit['Revision_1_OOF_robust_fail'] == True) | (df_case_audit['SNR_Only_3_8_robust_fail'] == True)]
print("
Speaker cases with robust failure in either Revision 1 or SNR-only 3/8:")
display(rob_fail_cases[['speaker_id', 'Revision_1_OOF_clean_exit', 'Revision_1_OOF_robust_fail', 'SNR_Only_3_8_clean_exit', 'SNR_Only_3_8_robust_fail']])


### Cell 11: Conditional Genuine Online GPU Timing Benchmark
Runs sequential inference on a single shared GPU instance with genuine early stopping in the teacher Euler trajectory.
Measures high-resolution wall-clock latency (ms) and Real-Time Factor (RTF) with CUDA synchronization before and after.

In [ ]:
# Cell 11: Conditional Genuine Online GPU Latency Benchmarking
import torch
from run_three_scheduler_revisions_dev import run_conditional_gpu_timing

df_timing = None
if torch.cuda.is_available() and CONFIG.get('run_online_timing', True):
    print("[Timing] CUDA GPU detected. Running sequential genuine online GPU latency benchmarking...")
    df_timing = run_conditional_gpu_timing(
        manifest_cases=manifest_cases,
        output_dir=CONFIG['output_dir'],
        device="cuda",
        timing_cases=CONFIG.get('timing_cases', 50),
    )
    if df_timing is not None:
        display(df_timing)
else:
    print("[Timing] CUDA unavailable or timing skipped. Latency marked as 'not measured' per protocol.")


### Cell 12: Export Master Report, Experiment Metadata & Downloadable Archive
Compiles `THREE_SCHEDULER_REVISIONS_DEVELOPMENT_REPORT.md` answering all 6 core questions, logs experiment metadata, and packages all 12 artifacts into `three_scheduler_revisions_dev_artifacts.tar.gz` for download.

In [ ]:
# Cell 12: Compile Master Report & Download Archive
import datetime
import tarfile
from run_three_scheduler_revisions_dev import generate_master_development_report, safe_json_serialize

# Generate Report
generate_master_development_report(
    df_all_pol=df_all_pol,
    df_paired=df_paired,
    df_folds=df_folds,
    df_timing=df_timing,
    val_report=val_report,
    output_dir=CONFIG['output_dir'],
)

# Export Metadata
meta = {
    "timestamp": datetime.datetime.now().isoformat(),
    "platform": platform.platform(),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__ if 'torch' in sys.modules else 'not_installed',
    "cuda_available": torch.cuda.is_available() if 'torch' in sys.modules else False,
    "n_development_speakers": 100,
    "n_cases": 200,
    "candidate_steps": CONFIG['candidate_steps'],
    "n_grid_records": len(df_grid),
}
with open(os.path.join(CONFIG['output_dir'], 'experiment_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, default=safe_json_serialize)

# Package Archive
archive_path = os.path.join(CONFIG['output_dir'], 'three_scheduler_revisions_dev_artifacts.tar.gz')
with tarfile.open(archive_path, 'w:gz') as tar:
    for fname in os.listdir(CONFIG['output_dir']):
        if fname != 'three_scheduler_revisions_dev_artifacts.tar.gz':
            tar.add(os.path.join(CONFIG['output_dir'], fname), arcname=fname)

print(f"[Archive Ready] {archive_path} ({os.path.getsize(archive_path) / 1024:.1f} KB)")

# Trigger Google Colab browser download if running in Colab
try:
    from google.colab import files
    files.download(archive_path)
    print("[Download] Archive download initiated in browser.")
except Exception as e:
    print(f"[Download Note] Running outside Google Colab. Artifact archive saved at {archive_path}")
